# Max Call Bermudean options pricing

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import time
import pandas as pd

# Plotting style
sns.set_theme()

# Select device: GPU if available, otherwise CPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")


Using device: cpu


### Parameters

In [ ]:
# Parameters
r = 0.05              # Risk-free rate
sigma = 0.2           # Volatility
dividend = 0.1        # Dividend Yield
x0 = 100              # Initial asset price
K = 100               # Strike price
rho = 0               # Assets correlation (Brownians correlation)
T = 3                 # Maturity (in years)
N = 9                 # Number of exercise dates

d=5                   # Asset numbers

M=

dt = T / N            # Time step

# For reproducibility
torch.manual_seed(42)
np.random.seed(42)


### Paths simulations

In [ ]:
def simulate_black_scholes_paths(x0, r, sigma, delta, T, N, M, d=5, device="cpu"):
    dt = T / N
    Z = torch.randn(N, M, d, device=device)

    x0 = torch.full((M, d), x0, device=device) if not torch.is_tensor(x0) else x0
    sigma = torch.full((d,), sigma, device=device) if not torch.is_tensor(sigma) else sigma
    delta = torch.full((d,), delta, device=device) if not torch.is_tensor(delta) else delta

    drift = (r - delta - 0.5 * sigma**2) * dt
    diffusion = sigma * torch.sqrt(torch.tensor(dt, device=device))

    increments = drift.unsqueeze(0).unsqueeze(0) + diffusion.unsqueeze(0).unsqueeze(0) * Z
    log_paths = torch.log(x0) + torch.cumsum(increments, dim=0)

    paths = torch.cat([
        x0.unsqueeze(0),
        torch.exp(log_paths)
    ], dim=0)

    return paths  # (N+1, M, d)


In [ ]:
import torch

def simulate_multidim_black_scholes_paths(
    x0, r, sigma, delta, T, N, M, d=5, rho=None, device="cpu"
):
    """
    Simule des trajectoires pour d actifs selon un modèle de Black-Scholes corrélé.

    x0 : float ou tensor (taille d) — prix initiaux
    r : float — taux sans risque
    sigma : float ou tensor (taille d) — volatilités
    delta : float ou tensor (taille d) — taux de dividende
    T : float — horizon temporel
    N : int — nombre de pas de temps
    M : int — nombre de trajectoires simulées
    d : int — nombre d'actifs (par défaut 5)
    rho : tensor (d,d) ou None — matrice de corrélation entre actifs
    device : str — "cpu" ou "cuda"
    """

    dt = T / N  # Pas de temps

    # Variables normales standard indépendantes
    Z = torch.randn(N, M, d, device=device)

    # Si une matrice de corrélation est donnée, la prendre en compte
    if rho is not None:
        # Vérifie que rho est un tensor
        if not torch.is_tensor(rho):
            rho = torch.tensor(rho, device=device, dtype=torch.float32)

        # Décomposition de Cholesky de la matrice de corrélation
        L = torch.linalg.cholesky(rho)  # rho = L @ L.T

        # Corrèle Z en appliquant la matrice L
        Z = torch.matmul(Z, L.T)  # (N, M, d) x (d, d) -> (N, M, d)

    # Force x0, sigma et delta à être des tensors
    def ensure_tensor(x, name):
        if not torch.is_tensor(x):
            return torch.full((d,), x, device=device, dtype=torch.float32)
        else:
            x_tensor = x.to(device=device, dtype=torch.float32)
            if x_tensor.ndim == 0:
                return x_tensor.expand(d)
            return x_tensor

    x0 = ensure_tensor(x0, "x0")
    sigma = ensure_tensor(sigma, "sigma")
    delta = ensure_tensor(delta, "delta")

    # Calcul du drift et de la diffusion
    drift = (r - delta - 0.5 * sigma**2) * dt  # (d,)
    diffusion = sigma * torch.sqrt(torch.tensor(dt, device=device))  # (d,)

    # Application du drift et de la diffusion
    increments = drift.unsqueeze(0).unsqueeze(0) + diffusion.unsqueeze(0).unsqueeze(0) * Z  # (N, M, d)

    # Log des prix cumulés
    log_x0 = torch.log(x0)
    log_paths = log_x0.unsqueeze(0).unsqueeze(0) + torch.cumsum(increments, dim=0)  # (N, M, d)

    # Exponentiation pour retrouver les prix
    paths = torch.cat([
        x0.unsqueeze(0).unsqueeze(0).expand(1, M, d),  # initial prices
        torch.exp(log_paths)
    ], dim=0)  # (N+1, M, d)

    return paths


In [ ]:
simulate_multidim_black_scholes_paths(x0, r, sigma, dividend, T, N, 10, d=5, rho=None, device="cpu")

tensor([[[100.0000, 100.0000, 100.0000, 100.0000, 100.0000],
         [100.0000, 100.0000, 100.0000, 100.0000, 100.0000],
         [100.0000, 100.0000, 100.0000, 100.0000, 100.0000],
         [100.0000, 100.0000, 100.0000, 100.0000, 100.0000],
         [100.0000, 100.0000, 100.0000, 100.0000, 100.0000],
         [100.0000, 100.0000, 100.0000, 100.0000, 100.0000],
         [100.0000, 100.0000, 100.0000, 100.0000, 100.0000],
         [100.0000, 100.0000, 100.0000, 100.0000, 100.0000],
         [100.0000, 100.0000, 100.0000, 100.0000, 100.0000],
         [100.0000, 100.0000, 100.0000, 100.0000, 100.0000]],

        [[122.0387, 115.9981, 108.4016,  76.6088, 105.6544],
         [ 84.7142,  97.2090,  81.1700,  89.5671, 118.1807],
         [ 93.3651,  83.0765,  89.8183,  91.5824,  89.3945],
         [106.6846, 118.0933,  95.9098,  92.2408, 102.7806],
         [ 89.5051, 110.6476, 107.1581, 118.6168, 113.2432],
         [113.4696, 104.8287, 113.9727,  95.1154,  98.1659],
         [ 94.8965, 10

In [ ]:
def bermudan_max_call_payoff(S, K):
    """
    Calcule le payoff d'une option Bermudéenne Max-Call.

    S : tensor (M, d) — Prix des actifs à la date d'exercice (M trajectoires, d actifs)
    K : float — Strike de l'option

    Retourne :
    payoff : tensor (M,) — Payoff pour chaque trajectoire
    """

    # Prend le maximum des actifs pour chaque trajectoire (sur la dimension des actifs)
    max_S = torch.max(S, dim=1).values  # shape (M,)

    # Payoff = (max_S - K)+
    payoff = torch.clamp(max_S - K, min=0.0)  # shape (M,)

    return payoff


In [ ]:
multi_dim_bs_path = simulate_multidim_black_scholes_paths(x0, r, sigma, dividend, T, N, 21000000, d=5, rho=None, device="cpu")

bermudan_max_call_payoff(multi_dim_bs_path[1], K)

In [ ]:
multi_dim_bs_path.shape

torch.Size([10, 21000000, 5])

In [ ]:
type(multi_dim_bs_path)

torch.Tensor

In [ ]:
def compute_all_payoffs(S_paths, K):
    """
    Calcule les payoffs à chaque pas de temps.
    S_paths : (N+1, M, d)
    Retourne : tensor (N+1, M)
    """
    N_plus_1, M, d = S_paths.shape
    payoffs = torch.zeros(N_plus_1, M, device=S_paths.device)

    for n in range(N_plus_1):
        payoffs[n] = bermudan_max_call_payoff(S_paths[n], K)

    return payoffs


In [ ]:
compute_all_payoffs(multi_dim_bs_path)

### Normalization

In [ ]:
def normalize(x):
    """
    Normalize input tensor x: return normalized x, mean, and std
    """
    x = x.to(device)
    mean = x.mean(0, keepdims=True)
    std = x.std(0, keepdims=True) + 1e-8  # to avoid division by 0
    return (x - mean) / std, mean, std

def unnormalize(x_norm, mean, std):
    """
    Reverse normalization given mean and std
    """
    return x_norm * std + mean


### Network architecture

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

# Fonction utilitaire : normalisation d'un batch
def normalize_batch(batch):
    """
    Normalise un batch de données (par feature)
    batch : tensor (batch_size, d)
    """
    mean = batch.mean(dim=0, keepdim=True)
    std = batch.std(dim=0, keepdim=True) + 1e-8  # pour éviter la division par 0
    return (batch - mean) / std

In [ ]:
# Définition du réseau de neurones pour la fonction de décision d'arrêt
class StoppingNet(nn.Module):
    def __init__(self, d, hidden_size=None):
        """
        d : dimension d'entrée (dimension de l'état)
        hidden_size : taille des couches cachées (par défaut 40+d)
        """
        super(StoppingNet, self).__init__()
        if hidden_size is None:
            hidden_size = 40 + d

        # Définition du modèle
        self.fc1 = nn.Linear(d, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        self.fc3 = nn.Linear(hidden_size, 1)

        # Initialisation Xavier
        nn.init.xavier_uniform_(self.fc1.weight)
        nn.init.xavier_uniform_(self.fc2.weight)
        nn.init.xavier_uniform_(self.fc3.weight)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = torch.sigmoid(self.fc3(x))  # Sortie entre (0,1)
        return x.squeeze(-1)  # Retourne (batch_size,)

In [ ]:

# Fonction d'entraînement des réseaux par backward induction
def train_stopping_networks(S_paths, payoffs, N, batch_size=8192, lr=1e-3, n_epochs=1):
    """
    Entraîne les réseaux de neurones pour l'optimal stopping.

    S_paths : tensor (N+1, M, d) — trajectoires simulées
    payoffs : tensor (N+1, M) — payoffs simulés à chaque pas de temps
    N : nombre total de pas de temps
    batch_size : taille des mini-batchs
    lr : learning rate
    n_epochs : nombre d'époques d'apprentissage pour chaque réseau

    Retourne :
    list_models : liste des modèles (de n=N-1 à n=1)
    """
    M, d = S_paths.shape[1], S_paths.shape[2]
    device = S_paths.device

    # Initialisation des réseaux
    models = [StoppingNet(d).to(device) for _ in range(N)]

    # Initialisation des temps d'arrêt connus : à N, on s'arrête toujours
    stopping_decision = torch.ones(M, device=device)

    for n in reversed(range(1, N)):
        model = models[n-1]
        optimizer = optim.Adam(model.parameters(), lr=lr)

        for epoch in range(n_epochs):
            perm = torch.randperm(M)
            for i in range(0, M, batch_size):
                idx = perm[i:i+batch_size]
                S_batch = S_paths[n, idx]  # prix au temps n
                S_batch = normalize_batch(S_batch)  # normalisation

                payoff_now = payoffs[n, idx]
                payoff_future = payoffs[stopping_decision[idx].long(), idx]

                # Forward
                stop_prob = model(S_batch)

                # Calcul de la loss
                loss = -torch.mean(
                    stop_prob * payoff_now + (1 - stop_prob) * payoff_future
                )

                # Backpropagation
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        # Mise à jour des décisions d'arrêt après entraînement
        with torch.no_grad():
            S_batch_full = normalize_batch(S_paths[n])
            stop_prob_full = model(S_batch_full)
            new_stop = stop_prob_full > 0.5  # décision dure
            # On met à jour les temps d'arrêt
            stopping_decision = torch.where(new_stop, torch.full_like(stopping_decision, n), stopping_decision)

    return models


In [ ]:
def train_stopping_networks(S_paths, payoffs, N, batch_size=8192, lr=1e-3, n_epochs=1, verbose=True):
    """
    Entraîne les réseaux de neurones pour l'optimal stopping.

    S_paths : tensor (N+1, M, d) — trajectoires simulées
    payoffs : tensor (N+1, M) — payoffs simulés à chaque pas de temps
    N : nombre total de pas de temps
    batch_size : taille des mini-batchs
    lr : learning rate
    n_epochs : nombre d'époques d'apprentissage pour chaque réseau
    verbose : bool — si True, affiche la loss pendant l'entraînement

    Retourne :
    list_models : liste des modèles (de n=N-1 à n=1)
    loss_history : liste des pertes pour chaque réseau
    """
    M, d = S_paths.shape[1], S_paths.shape[2]
    device = S_paths.device

    # Initialisation des réseaux
    models = [StoppingNet(d).to(device) for _ in range(N)]

    # Stocke l'historique des pertes
    loss_history = [[] for _ in range(N)]

    # Initialisation des temps d'arrêt connus : à N, on s'arrête toujours
    stopping_decision = torch.ones(M, device=device)

    for n in reversed(range(1, N)):
        model = models[n-1]
        optimizer = optim.Adam(model.parameters(), lr=lr)

        if verbose:
            print(f"Training model for time step {n}...")

        for epoch in range(n_epochs):
            perm = torch.randperm(M)
            batch_losses = []
            for i in range(0, M, batch_size):
                idx = perm[i:i+batch_size]
                S_batch = S_paths[n, idx]
                S_batch = normalize_batch(S_batch)

                payoff_now = payoffs[n, idx]
                payoff_future = payoffs[stopping_decision[idx].long(), idx]

                # Forward pass
                stop_prob = model(S_batch)

                # Calcul de la loss
                loss = -torch.mean(
                    stop_prob * payoff_now + (1 - stop_prob) * payoff_future
                )

                # Backpropagation
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

                # Stocke la perte pour ce batch
                batch_losses.append(loss.item())

                if verbose:
                    print(f"Time step {n}, Epoch {epoch+1}, Batch {i//batch_size+1}, Loss: {loss.item():.6f}")

            # Enregistre la perte moyenne sur cette époque
            loss_history[n-1].append(sum(batch_losses) / len(batch_losses))

          # Mise à jour des décisions d'arrêt après entraînement
        with torch.no_grad():
            S_batch_full = normalize_batch(S_paths[n])
            stop_prob_full = model(S_batch_full)
            new_stop = stop_prob_full > 0.5  # décision dure
            # On met à jour les temps d'arrêt
            stopping_decision = torch.where(new_stop, torch.full_like(stopping_decision, n), stopping_decision)

    return models, loss_history


In [ ]:
import matplotlib.pyplot as plt

models, loss_history = train_stopping_networks(S_paths, payoffs, N, batch_size=8192, lr=1e-3, n_epochs=5)

for n, losses in enumerate(loss_history):
    plt.plot(losses, label=f'Time step {n+1}')

plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss Evolution for Each Time Step')
plt.legend()
plt.show()


In [ ]:
# Fonction utilitaire : lissage avec moyenne glissante
def moving_average(values, window=5):
    """
    Calcule la moyenne glissante d'une liste de valeurs.
    values : list of floats
    window : taille de la fenêtre
    """
    if len(values) < window:
        return values  # pas assez de points pour lisser
    return [sum(values[i-window:i])/window for i in range(window, len(values)+1)]


In [ ]:
# Simuler
S_paths = simulate_black_scholes_paths(x0, r, sigma, dividend, T, N, M, d=d, device=device)

# Calculer les payoffs
payoffs = compute_all_payoffs(S_paths, K)

# Entraîner
models, loss_history = train_stopping_networks(S_paths, payoffs, N, batch_size=8192, lr=1e-3, n_epochs=10)

for n, losses in enumerate(loss_history):
    smoothed_losses = moving_average(losses, window=3)
    plt.plot(smoothed_losses, label=f'Time step {n+1}')

plt.xlabel('Epoch')
plt.ylabel('Smoothed Loss')
plt.title('Smoothed Loss Evolution for Each Time Step')
plt.legend()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

models, loss_history = train_stopping_networks(S_paths, payoffs, N, batch_size=8192, lr=1e-3, n_epochs=5)

for n, losses in enumerate(loss_history):
    smoothed_losses = moving_average(losses, window=3)  # lissage sur 3 epochs
    plt.plot(smoothed_losses, label=f'Time step {n+1}')

plt.xlabel('Epoch')
plt.ylabel('Smoothed Loss')
plt.title('Smoothed Loss Evolution for Each Time Step')
plt.legend()
plt.show()


In [ ]:
# class StoppingNetwork(nn.Module):
#     def __init__(self, input_dim=1, hidden_dim=32):
#         super().__init__()
#         self.net = nn.Sequential(
#             nn.Linear(input_dim, hidden_dim),
#             nn.ReLU(),

#             nn.Linear(hidden_dim, hidden_dim),
#             nn.ReLU(),

#             nn.Linear(hidden_dim, hidden_dim),
#             nn.ReLU(),

#             nn.Linear(hidden_dim, 1),
#             nn.Sigmoid()
#         )

#     def forward(self, x):
#         x = x.to(device).float()
#         return self.net(x).squeeze(-1)


### Weights Initialization

In [ ]:
# def init_weights(m):
#     if isinstance(m, nn.Linear):
#         torch.nn.init.xavier_uniform_(m.weight)
#         if m.bias is not None:
#             torch.nn.init.zeros_(m.bias)

# net_Nm1 = StoppingNetwork(input_dim=1, hidden_dim=32).to(device)
# net_Nm1.apply(init_weights)


StoppingNetwork(
  (net): Sequential(
    (0): Linear(in_features=1, out_features=32, bias=True)
    (1): ReLU()
    (2): Linear(in_features=32, out_features=32, bias=True)
    (3): ReLU()
    (4): Linear(in_features=32, out_features=32, bias=True)
    (5): ReLU()
    (6): Linear(in_features=32, out_features=1, bias=True)
    (7): Sigmoid()
  )
)

### Stochastic gradient ascent

In [ ]:
# # --- Custom loss function: stochastic gradient ascent ---
# class StopGainLoss(nn.Module):
#     def forward(self, stop_prob, payoff_now, payoff_next):
#         """
#         stop_prob: output of the network, shape (batch_size,)
#         payoff_now: g(n, X_n), shape (batch_size,)
#         payoff_next: g(n+1, X_{n+1}), shape (batch_size,)
#         """
#         # Stochastic gradient ascent: maximize expected reward
#         # In PyTorch, we minimize => return negative
#         expected_reward = stop_prob * payoff_now + (1 - stop_prob) * payoff_next
#         return -torch.mean(expected_reward)


### Training

In [ ]:
# M_train=5000

# # --- Simulate trajectories for training
# sample_paths = simulate_black_scholes_paths(x0, r, sigma, T, N, M_train)

# # payoffs
# payoffs = np.maximum(K - sample_paths, 0)

# n_epochs = 20
# batch_size = 1024

# # To store trained networks and normalization stats
# trained_nets = {}
# normalizers = {}  # stores (x_mean, x_std, y_mean, y_std)

# trained_nets[N] = lambda x: torch.ones(x.size(0), device=x.device)  # f_N(x) = 1
# normalizers[N] = (torch.tensor([0.0], device=device), torch.tensor([1.0], device=device),
#                   torch.tensor([0.0], device=device), torch.tensor([1.0], device=device))


# for n in reversed(range(N)):
#     print(f"\nTraining f_{n}...")

#     S_n = sample_paths[n]
#     X_n = S_n.view(-1, 1).float().to(device)


#     if n == N - 1:
#         y_n = payoffs[N].float().to(device)
#         y_continuation = y_n
#     else:
#         S_np1 = sample_paths[n + 1]
#         S_np1_tensor = S_np1.view(-1, 1).float().to(device)


#         x_mean_next, x_std_next, y_mean_next, y_std_next = normalizers[n + 1]
#         S_np1_norm = (S_np1_tensor - x_mean_next) / x_std_next

#         with torch.no_grad():
#             cont_prob = trained_nets[n + 1](S_np1_norm)

#         y_continuation = unnormalize(cont_prob.view(-1, 1), y_mean_next, y_std_next).view(-1).to(device)

#     reward_n = payoffs[n]
#     y_n = torch.max(reward_n, y_continuation)

#     # Normalisation
#     X_n_norm, x_mean, x_std = normalize(X_n)
#     y_n_norm, y_mean, y_std = normalize(y_n.view(-1, 1))
#     y_n_norm = y_n_norm.view(-1)

#     y_cont_norm = ((y_continuation - y_mean) / y_std).view(-1)

#     normalizers[n] = (x_mean, x_std, y_mean, y_std)

#     # training
#     net_n = StoppingNetwork(input_dim=1, hidden_dim=32).to(device)
#     net_n.apply(init_weights)
#     optimizer = optim.Adam(net_n.parameters(), lr=1e-3)
#     loss_fn = StopGainLoss()

#     for epoch in range(n_epochs):
#         idx = torch.randperm(M_train)
#         X_shuffled = X_n_norm[idx]
#         y_shuffled = y_n_norm[idx]
#         y_cont_shuffled = y_cont_norm[idx]

#         for i in range(0, M_train, batch_size):
#             xb = X_shuffled[i:i+batch_size]
#             yb = y_shuffled[i:i+batch_size]
#             yb_next = y_cont_shuffled[i:i+batch_size]

#             pred = net_n(xb)
#             loss = loss_fn(pred, yb, yb_next)


#             optimizer.zero_grad()
#             loss.backward()
#             optimizer.step()

#     trained_nets[n] = net_n



Training f_9...

Training f_8...

Training f_7...

Training f_6...

Training f_5...

Training f_4...

Training f_3...

Training f_2...

Training f_1...

Training f_0...


### Lower bound

In [ ]:
def compute_lower_bound(trained_nets, normalizers, x0, r, sigma, K, T, N, M_eval):
    """
    Compute the lower bound estimate using the learned stopping policy.
    """
    # Generate independent evaluation sample paths (on GPU)
    sample_eval = simulate_black_scholes_paths(x0, r, sigma, T, N, M_eval, device=device)
    payoff_eval = torch.maximum(K - sample_eval, torch.tensor(0.0, device=device))

    # Init rewards and stopping indicators
    rewards = torch.zeros(M_eval, device=device)
    stopping_times = torch.full((M_eval,), N, device=device)

    for n in range(N):
        S_n_tensor = sample_eval[n].view(-1, 1).float().to(device)
        x_mean, x_std, _, _ = normalizers[n]
        S_n_norm = (S_n_tensor - x_mean) / x_std

        with torch.no_grad():
            stop_prob = trained_nets[n](S_n_norm)

        should_stop = (stop_prob > 0.5) & (stopping_times == N)
        rewards[should_stop] = payoff_eval[n][should_stop]
        stopping_times[should_stop] = n

    # Payoff at maturity for those never stopped
    rewards[stopping_times == N] = payoff_eval[N][stopping_times == N]

    # Return to CPU for final stats
    rewards_cpu = rewards.cpu().numpy()
    return np.mean(rewards_cpu), np.std(rewards_cpu, ddof=1), M_eval


In [ ]:
# Set number of Monte Carlo paths for evaluation (pricing)
M_eval = 10000

# Call the function to compute the lower bound
L_hat, sigma_L, K_L = compute_lower_bound(
    trained_nets=trained_nets,
    normalizers=normalizers,
    x0=x0,
    r=r,
    sigma=sigma,
    K=K,
    T=T,
    N=N,
    M_eval=M_eval
)


# Display the result
print(f"Estimated lower bound L̂: {L_hat:.4f}")
print(f"Estimated std: {sigma_L:.4f}")


Estimated lower bound L̂: 3.4215
Estimated std: 4.8898


### Upper bound with Richardson extrapolation

In [ ]:
def compute_dual_upper_bound_richardson(trained_nets, normalizers, x0, r, sigma, K, T, N, M_dual=500, J=100):
    """
    Estimates the dual upper bound using Richardson extrapolation.
    Fully vectorized version for speed: J sub-paths per main path are simulated in batch.
    """

    def estimate_continuation(S_n, n):
        """
        Vectorized estimation of continuation using J sub-trajectories per S_n[i].
        Returns: tensor of shape (M,)
        """
        if n == N:
            return torch.zeros(S_n.shape[0], device=device)

        M = S_n.shape[0]
        T_remain = T * (1 - n / N)
        N_remain = N - n

        # Repeat S_n J times to build (N_remain + 1, M*J) paths
        S0 = S_n.repeat(J, 1).T.flatten()  # shape (M * J,)

        sub_paths = simulate_black_scholes_paths(S0, r, sigma, T_remain, N_remain, M * J, device=device)
        sub_payoffs = torch.clamp(K - sub_paths, min=0)  # shape (N_remain + 1, M * J)

        # Determine stopping times
        tau = torch.full((M * J,), N_remain, device=device)

        for t in range(N_remain):
            S_t = sub_paths[t].view(M * J, 1)
            x_mean, x_std, *_ = normalizers[n + t]
            S_t_norm = (S_t - x_mean) / x_std
            with torch.no_grad():
                stop_prob = trained_nets[n + t](S_t_norm)

            # Find where stopping occurs
            mask = (tau == N_remain) & (stop_prob > 0.5)
            tau[mask] = t

        # Gather the corresponding payoffs at stopping times
        indices = tau + torch.arange(0, M * J, device=device) * (N_remain + 1)
        flat_payoffs = sub_payoffs.flatten()
        rewards = flat_payoffs[indices]

        # Reshape to (M, J) then average over J for each main path
        rewards = rewards.view(M, J)
        return rewards.mean(dim=1)

    # --- Coarse path (N steps) ---
    sample_N = simulate_black_scholes_paths(x0, r, sigma, T, N, M_dual, device=device)
    payoffs_N = torch.clamp(K - sample_N, min=0)
    M_N = torch.zeros((N + 1, M_dual), device=device)
    C_N = torch.zeros((N + 1, M_dual), device=device)

    for n in range(1, N + 1):
        S_n = sample_N[n].view(-1, 1)
        x_mean, x_std, *_ = normalizers[n - 1]
        S_norm = (S_n - x_mean) / x_std
        with torch.no_grad():
            stop_prob = trained_nets[n - 1](S_norm)
        g_n = payoffs_N[n]
        C_est = estimate_continuation(S_n, n)
        C_N[n] = stop_prob * g_n + (1 - stop_prob) * C_est
        M_N[n] = M_N[n - 1] + (C_N[n] - C_N[n - 1])

    U_N = torch.max(payoffs_N - M_N, dim=0).values

    # --- Fine path (2N steps) ---
    N2 = 2 * N
    sample_2N = simulate_black_scholes_paths(x0, r, sigma, T, N2, M_dual, device=device)
    payoffs_2N = torch.clamp(K - sample_2N, min=0)
    M_2N = torch.zeros((N + 1, M_dual), device=device)
    C_2N = torch.zeros((N + 1, M_dual), device=device)

    for n in range(1, N + 1):
        idx = 2 * n
        S_n = sample_2N[idx].view(-1, 1)
        x_mean, x_std, *_ = normalizers[n - 1]
        S_norm = (S_n - x_mean) / x_std
        with torch.no_grad():
            stop_prob = trained_nets[n - 1](S_norm)
        g_n = payoffs_2N[idx]
        C_est = estimate_continuation(S_n, n)
        C_2N[n] = stop_prob * g_n + (1 - stop_prob) * C_est
        M_2N[n] = M_2N[n - 1] + (C_2N[n] - C_2N[n - 1])

    U_2N = torch.max(payoffs_2N[::2] - M_2N, dim=0).values

    # Richardson extrapolation
    U_rich = 2 * U_2N - U_N
    return U_rich.mean().item(), U_rich.std(unbiased=True).item(), len(U_rich)


In [ ]:
U_hat, sigma_U, K_U = compute_dual_upper_bound_richardson(
    trained_nets, normalizers,
    x0=x0, r=r, sigma=sigma, K=K, T=T, N=N,
    M_dual=2000, J=100
)

print(f"Estimated upper bound: {U_hat:.4f}")
print(f"Standard deviation: {sigma_U:.4f}")

Estimated upper bound: 2.3038
Standard deviation: 6.7588


## Upper Bound without Richardson extrapolation

In [ ]:
def compute_dual_upper_bound_no_richardson(trained_nets, normalizers, x0, r, sigma, K, T, N, M_dual=10000, J=300):
    """
    Compute the dual upper bound without Richardson extrapolation.
    Includes rigorous martingale update using sub-trajectories for continuation.
    """
    def estimate_continuation(S_n, n):
        """
        Vectorized estimation of continuation using J sub-trajectories per S_n[i].
        Returns: tensor of shape (M,)
        """
        if n == N:
            return torch.zeros(S_n.shape[0], device=device)

        M = S_n.shape[0]
        T_remain = T * (1 - n / N)
        N_remain = N - n

        # Repeat S_n J times to build (N_remain + 1, M*J) paths
        S0 = S_n.repeat(J, 1).T.flatten()  # shape (M * J,)

        sub_paths = simulate_black_scholes_paths(S0, r, sigma, T_remain, N_remain, M * J, device=device)
        sub_payoffs = torch.clamp(K - sub_paths, min=0)  # shape (N_remain + 1, M * J)

        # Determine stopping times
        tau = torch.full((M * J,), N_remain, device=device)

        for t in range(N_remain):
            S_t = sub_paths[t].view(M * J, 1)
            x_mean, x_std, *_ = normalizers[n + t]
            S_t_norm = (S_t - x_mean) / x_std
            with torch.no_grad():
                stop_prob = trained_nets[n + t](S_t_norm)

            # Find where stopping occurs
            mask = (tau == N_remain) & (stop_prob > 0.5)
            tau[mask] = t

        # Gather the corresponding payoffs at stopping times
        indices = tau + torch.arange(0, M * J, device=device) * (N_remain + 1)
        flat_payoffs = sub_payoffs.flatten()
        rewards = flat_payoffs[indices]

        # Reshape to (M, J) then average over J for each main path
        rewards = rewards.view(M, J)
        return rewards.mean(dim=1)

    # --- Coarse path (N steps) ---
    sample_N = simulate_black_scholes_paths(x0, r, sigma, T, N, M_dual, device=device)
    payoffs_N = torch.clamp(K - sample_N, min=0)
    M_N = torch.zeros((N + 1, M_dual), device=device)
    C_N = torch.zeros((N + 1, M_dual), device=device)

    for n in range(1, N + 1):
        S_n = sample_N[n].view(-1, 1)
        x_mean, x_std, *_ = normalizers[n - 1]
        S_norm = (S_n - x_mean) / x_std
        with torch.no_grad():
            stop_prob = trained_nets[n - 1](S_norm)
        g_n = payoffs_N[n]
        C_est = estimate_continuation(S_n, n)
        C_N[n] = stop_prob * g_n + (1 - stop_prob) * C_est
        M_N[n] = M_N[n - 1] + (C_N[n] - C_N[n - 1])

    U_N = torch.max(payoffs_N - M_N, dim=0).values

    return U_N.mean().item(), U_N.std(unbiased=True).item(), len(U_N)


In [ ]:
U_hat, sigma_U, K_U = compute_dual_upper_bound_no_richardson(
    trained_nets, normalizers,
    x0=x0, r=r, sigma=sigma, K=K, T=T, N=N,
    M_dual=2000, J=100
)

print(f"Estimated upper bound: {U_hat:.4f}")
print(f"Standard deviation: {sigma_U:.4f}")

Estimated upper bound: 1.8787
Standard deviation: 2.9770


### Confident Interval

In [ ]:

def confidence_interval(L_hat, U_hat, K_L, K_U, sigma_U, sigma_L, alpha=0.05):
    """
    Compute confidence interval for V0 following the paper (unilateral bounds).
    Returns the midpoint estimate and the interval:
        [L̂ - z * σ̂_L / √K_L, Û + z * σ̂_U / √K_U]    """

    z = stats.norm.ppf(1 - alpha / 2)

    ci_low = L_hat - z * sigma_L / np.sqrt(K_L)
    ci_up  = U_hat + z * sigma_U / np.sqrt(K_U)

    point_estimate = 0.5 * (L_hat + U_hat)
    return point_estimate, (ci_low, ci_up)


In [ ]:
# Compute confidence interval
V0_hat, (ci_lower, ci_upper) = confidence_interval(L_hat, U_hat, K_L, K_U, sigma_U, sigma_L, alpha=0.05)

# Display results
print(f"Point estimate of option price: {V0_hat:.4f}")
print(f"95% Confidence interval: [{ci_lower:.4f}, {ci_upper:.4f}]")

Point estimate of option price: 2.6501
95% Confidence interval: [3.3257, 2.0091]


#### First results

In [ ]:
S0_values = [80, 90, 100, 110, 120]

results = []
M_d = 5000   # dual paths
M_eval = 50000  # primal paths (lower bound)
J = 50

for idx, S0_test in enumerate(S0_values):
    print(f"\n--- Evaluating for S0 = {S0_test} ---")
    start_time = time.time()

    # Lower bound
    L_hat, sigma_L, K_L = compute_lower_bound(
        trained_nets=trained_nets,
        normalizers=normalizers,
        x0=S0_test,
        r=r,
        sigma=sigma,
        K=K,
        T=T,
        N=N,
        M_eval=M_eval
    )

    # Upper bound
    U_hat, sigma_U, K_U = compute_dual_upper_bound_richardson(
        trained_nets=trained_nets,
        normalizers=normalizers,
        x0=S0_test,
        r=r,
        sigma=sigma,
        K=K,
        T=T,
        N=N,
        M_dual=M_d,
        J=J
    )

    # IC 95%
    V0_hat, (ci_low, ci_up) = confidence_interval(
        L_hat, U_hat, K_L, K_U, sigma_U, sigma_L, alpha=0.05
    )

    elapsed = round(time.time() - start_time, 2)

    results.append({
        "S0": S0_test,
        "Lower_Bound": L_hat,
        "Upper_Bound": U_hat,
        "Point_Estimate": V0_hat,
        "95% CI": (ci_low, ci_up),
        "Std Lower": sigma_L,
        "Std Upper": sigma_U,
        "Time (s)": elapsed
    })

# Affichage des résultats sous forme de DataFrame
df_results = pd.DataFrame(results)
print(df_results)



--- Evaluating for S0 = 80 ---

--- Evaluating for S0 = 90 ---

--- Evaluating for S0 = 100 ---

--- Evaluating for S0 = 110 ---

--- Evaluating for S0 = 120 ---
    S0  Lower_Bound  Upper_Bound  Point_Estimate  \
0   80    19.773623    20.000799       19.887211   
1   90    10.219204    10.178594       10.198899   
2  100     3.430863     2.082746        2.756805   
3  110     0.000000     1.145346        0.572673   
4  120     0.000000     0.582811        0.291405   

                                    95% CI  Std Lower  Std Upper  Time (s)  
0  (19.71115864583618, 20.003392629635073)   7.126328   0.093565      3.35  
1  (10.157106911645327, 10.23103621203248)   7.084477   1.891999      3.25  
2  (3.388208870539549, 2.2731373709070355)   4.866333   6.868860      3.17  
3                (0.0, 1.2768030628802232)   0.000000   4.742646      3.22  
4                 (0.0, 0.666354553347589)   0.000000   3.014061      3.14  


### Richardson extrapolation impact on convergence and simulation

In [ ]:
'''K_samples = 20
U_vals_rich = []
U_vals_no_rich = []
times_rich = []
times_no_rich = []

for _ in range(K_samples):
    # With Richardson
    start_rich = time.time()
    U_rich = compute_dual_upper_bound_richardson(trained_nets, normalizers, M_dual=50000)
    end_rich = time.time()

    U_vals_rich.append(U_rich)
    times_rich.append(end_rich - start_rich)

    # Without Richardson
    start_no_rich = time.time()
    U_no_rich = compute_dual_upper_bound_no_richardson(trained_nets, normalizers, M_dual=50000)
    end_no_rich = time.time()

    U_vals_no_rich.append(U_no_rich)
    times_no_rich.append(end_no_rich - start_no_rich)

# Build DataFrame with results
df_compare = pd.DataFrame({
    "Method": ["With Richardson", "Without Richardson"],
    "Mean": [np.mean(U_vals_rich), np.mean(U_vals_no_rich)],
    "Variance": [np.var(U_vals_rich, ddof=1), np.var(U_vals_no_rich, ddof=1)],
    "Avg Time (s)": [np.mean(times_rich), np.mean(times_no_rich)]
})


print(df_compare)


SyntaxError: incomplete input (3307450724.py, line 1)

While Richardson extrapolation requires more computational time, it generally produces a tighter (i.e., lower) upper bound estimate. This indicates that it introduces less bias and provides a more accurate approximation of the true option value. However, this gain in precision comes at the cost of increased variance. In summary, Richardson extrapolation offers a bias-variance trade-off: lower bias but higher variability, whereas the standard method is faster and more stable but slightly less accurate.